# MetaDrive 小车键盘控制

运行下面的代码单元后，可以使用以下按键控制小车：

- `I` / `W` / `↑`：前进
- `K` / `S` / `↓`：制动或倒车
- `J` / `A` / `←`：左转
- `L` / `D` / `→`：右转
- `Esc`：结束运行并关闭仿真环境

按键可以组合使用，例如同时按住 `I` 和 `J` 向左前方行驶。

In [ ]:
import logging
import threading
import time

import numpy as np
from pynput import keyboard
from metadrive import MetaDriveEnv
from metadrive.component.map.base_map import BaseMap
from metadrive.component.map.pg_map import MapGenerateMethod
from metadrive.component.sensors.rgb_camera import RGBCamera
from metadrive.utils.doc_utils import generate_gif


# pynput 在独立线程中更新按键集合，因此用锁保护共享状态。
current_keys = set()
key_lock = threading.Lock()
stop_event = threading.Event()

SPECIAL_KEYS = {
    keyboard.Key.up: "up",
    keyboard.Key.down: "down",
    keyboard.Key.left: "left",
    keyboard.Key.right: "right",
}
CONTROL_KEYS = {"i", "j", "k", "l", "w", "a", "s", "d", "up", "down", "left", "right"}


def normalize_key(key):
    """将普通字符键和方向键转换为统一的字符串名称。"""
    char = getattr(key, "char", None)
    if char is not None:
        return char.lower()
    return SPECIAL_KEYS.get(key)


def on_press(key):
    if key == keyboard.Key.esc:
        stop_event.set()
        return False

    key_name = normalize_key(key)
    if key_name in CONTROL_KEYS:
        with key_lock:
            current_keys.add(key_name)


def on_release(key):
    key_name = normalize_key(key)
    if key_name is not None:
        with key_lock:
            current_keys.discard(key_name)


def read_action():
    """根据当前按键返回 MetaDrive 的 [转向, 油门/刹车] 动作。"""
    with key_lock:
        keys = current_keys.copy()

    forward = bool(keys & {"i", "w", "up"})
    reverse = bool(keys & {"k", "s", "down"})
    turn_left = bool(keys & {"j", "a", "left"})
    turn_right = bool(keys & {"l", "d", "right"})

    # MetaDrive 中：正转向值表示左转，正油门值表示加速。
    steer = float(turn_left) - float(turn_right)
    throttle_brake = float(forward) - float(reverse)
    return [steer, throttle_brake]


map_config = {
    BaseMap.GENERATE_TYPE: MapGenerateMethod.BIG_BLOCK_SEQUENCE,
    BaseMap.GENERATE_CONFIG: "SCS",  # 3 个路段
    BaseMap.LANE_WIDTH: 4,
    BaseMap.LANE_NUM: 1,
}


if __name__ == "__main__":
    config = dict(
        use_render=True,
        manual_control=False,  # 动作由下面的 env.step(action) 传入
        traffic_density=0.0,
        num_scenarios=10000,
        random_agent_model=False,
        on_continuous_line_done=True,
        out_of_route_done=True,
        image_observation=True,
        # MetaDrive 在 CPU 模式下会对任一边超过 100 像素的传感器缓冲区发出警告。
        # 96×54 保持 16:9 比例，也足够用于演示 GIF。
        sensors=dict(rgb_camera=(RGBCamera, 96, 54)),
        norm_pixel=True,
        # Jupyter 会交错显示 MetaDrive/Panda3D 的多线程 INFO 日志；只保留警告和错误。
        log_level=logging.WARNING,
        vehicle_config=dict(
            show_lidar=False,
            show_navi_mark=False,
            show_line_to_navi_mark=False,
        ),
        map_config=map_config,
    )

    env = MetaDriveEnv(config)
    listener = keyboard.Listener(on_press=on_press, on_release=on_release)
    frames = []

    try:
        observation, _ = env.reset(seed=21)
        start = time.time()
        listener.start()
        print("键盘控制已启动：I/J/K/L（也支持 W/A/S/D 和方向键），按 Esc 退出。")

        while not stop_event.is_set():
            action = read_action()
            observation, reward, terminated, truncated, info = env.step(action)

            image = observation["image"][..., -1]
            # norm_pixel=True 时图像范围为 [0, 1]；生成 GIF 前恢复为 uint8 [0, 255]。
            image_uint8 = np.clip(image * 255.0, 0, 255).astype(np.uint8)
            frames.append(image_uint8[..., ::-1])

            env.render(
                {
                    "Keyboard Control": "I/J/K/L or W/A/S/D",
                    "Action [steer, throttle]": str(action),
                    "Exit": "Esc",
                }
            )

            if info.get("arrive_dest", False):
                print("到达终点", info.get("episode_length"), "得分", info.get("episode_reward"))
                print("花费时间 {:.2f} 秒".format(time.time() - start))
                generate_gif(frames, gif_name="manual-control.gif")
                break

            if terminated or truncated:
                print("本轮结束，重新开始")
                observation, _ = env.reset(seed=env.current_seed)
                frames.clear()
                start = time.time()
    finally:
        stop_event.set()
        listener.stop()
        if listener.is_alive():
            listener.join(timeout=1.0)
        env.close()
        print("仿真环境已关闭。")
